# ARTI 402 — Deep Learning
## Lab 2 — Activations, Loss, and How a Network Learns

Activation functions and feedforward neural networks; gradient-based optimisation and back-propagation.

| | |
|---|---|
| **Marks** | **1 mark** (graded) |
| **Estimated time** | 100–120 minutes |
| **Prerequisites** | Lab 1 — you should be comfortable with `np.dot`, shapes, and the forward pass |

---

### How to work through the notebook

* Sections are labelled **Idea** (read and run), **Exercise** (you write code) and **Checkpoint** (a short answer).
* Every exercise cell is marked `# TODO`. Do not delete the cells above it — later cells depend on them.
* Run cells **in order**, top to bottom. If something breaks, restart the kernel and run all.
* The graded **Assessment** is at the very end.

---

### By the end of this lab you should be able to

1. Demonstrate, numerically, that stacked linear layers collapse into a single layer.
2. Implement **ReLU**, **sigmoid** and **softmax**, and choose the right one for a given layer.
3. Build a reusable `Layer_Dense` class and stack it into a feedforward network.
4. Measure how wrong a network is using **categorical cross-entropy** loss.
5. Estimate a derivative numerically and use it to run **gradient descent**.
6. Hand-compute a **backward pass** through a single neuron using the chain rule.


---
## Setup

Run this cell first. The `vertical_data` function builds a small toy dataset for you — three
clusters of points in 2D. You will use it in the Assessment. No downloads needed.


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)


def vertical_data(samples, classes):
    """Toy dataset: `classes` vertical clusters of `samples` points each.

    Returns X of shape (samples*classes, 2) and y of shape (samples*classes,).
    """
    X = np.zeros((samples * classes, 2))
    y = np.zeros(samples * classes, dtype="uint8")
    for class_number in range(classes):
        ix = range(samples * class_number, samples * (class_number + 1))
        X[ix] = np.c_[np.random.randn(samples) * 0.1 + class_number / 3,
                      np.random.randn(samples) * 0.1 + 0.5]
        y[ix] = class_number
    return X, y


print("\nSetup OK")

---
## Idea 1 — Proving the collapse

At the end of Lab 1 you had this network:

```
X (3, 4)  -->  layer 1: 5 neurons  -->  layer 2: 3 neurons  -->  scores
```

and the claim was that the two layers could be replaced by **one**.

Here is the algebra, and it is three lines. With zero biases, layer 1 computes `X @ W1.T` and
layer 2 computes `(X @ W1.T) @ W2.T`. Matrix multiplication is associative, so:

$$(X W_1^{\top}) W_2^{\top} = X (W_1^{\top} W_2^{\top}) = X (W_2 W_1)^{\top}$$

So a **single** weight matrix $W_{eq} = W_2 W_1$ does the identical job. Two layers, one layer,
same numbers. The depth is decoration.

Let's prove it with the exact network from Lab 1.


In [ ]:
# The same network you built in Lab 1's assessment.
X = np.array([[ 1.0,  2.0,  3.0,  2.5],
              [ 2.0,  5.0, -1.0,  2.0],
              [-1.5,  2.7,  3.3, -0.8]])

rng = np.random.default_rng(402)
W1 = rng.normal(0, 0.5, size=(5, 4))
b1 = np.zeros(5) # list of 5 zeros
W2 = rng.normal(0, 0.5, size=(3, 5))
b2 = np.zeros(3) # list of 3 zeros

# The two-layer forward pass from Lab 1.
layer1_out = np.dot(X, W1.T) + b1
two_layer  = np.dot(layer1_out, W2.T) + b2

print("two-layer output:")
print(two_layer)

### Exercise 1 — collapse the network

Build the single equivalent layer and show it produces the same numbers.


In [ ]:
# TODO: build the single equivalent weight matrix. Hint: W_eq = W2 @ W1
#       (@ is matrix multiplication - the same thing as np.dot for 2-D arrays)
W_eq = ...

# TODO: run the ONE-layer forward pass using W_eq
one_layer = ...

print("W1 shape :", W1.shape, " W2 shape:", W2.shape)
print("W_eq shape:", W_eq.shape, "  <- one layer, 4 inputs -> 3 outputs")
print()
print("one-layer output:")
print(one_layer)

# --- self-check ---
assert W_eq.shape == (3, 4), f"expected (3, 4), got {W_eq.shape}"
assert np.allclose(one_layer, two_layer), \
    "the two outputs should be identical - check the order of W2 and W1"
print("\nExercise 1 passed - the outputs are identical to the last decimal.")

Stop and take that in. Those 43 parameters were doing the work of 15.

This is not a quirk of these particular numbers. It is true for **any** stack of purely linear
layers, of any depth. A hundred linear layers still collapse to one.

The fix is to put something **non-linear** between the layers. That is what an activation
function is for.


---
## Idea 2 — The four activation functions you need

An activation function is applied to a layer's output, element by element. Its whole job is to
introduce a **bend**, so that stacking layers actually buys you something.

| Name | Formula | Output range | Where you use it |
|---|---|---|---|
| **Step** | 1 if x > 0 else 0 | {0, 1} | Historical only. Nobody trains with it. |
| **Linear** | x | any | Output layer of a **regression** model |
| **Sigmoid** | 1 / (1 + e^(-x)) | (0, 1) | Output layer for **binary** classification |
| **ReLU** | max(0, x) | [0, ∞) | **Hidden layers** — the default choice |

### Why the step function lost

The step function is either 0 or 1, and nothing else. An input of 3 and an input of 300,000 give
exactly the same answer. When you later ask "would nudging this weight have helped?", the step
function has no useful reply — it either flips or it doesn't. There is no sense of *how close*
it was. That makes it almost useless for training.

### Why sigmoid replaced it, and then lost too

Sigmoid is smooth. It squashes everything into (0, 1) but keeps the ordering and the magnitude of the input, so "how close was it" has a real answer.

Then people noticed two things. It is expensive to compute, and — more importantly — for large positive or large negative inputs it goes almost perfectly flat, which causes problems during training.

### Why ReLU won

`max(0, x)`. That is it. It is a single comparison, so it is extremely fast. It is *almost* linear, which turns out to be enough: that one bend at zero is all the non-linearity you need. Today it is the default for hidden layers in essentially every architecture.

Run the cell to see all four.

In [ ]:
x = np.linspace(-5, 5, 200)

funcs = [
    ("Step",    np.where(x > 0, 1, 0)),
    ("Linear",  x),
    ("Sigmoid", 1 / (1 + np.exp(-x))),
    ("ReLU",    np.maximum(0, x)),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, (name, y_vals) in zip(axes, funcs):
    ax.plot(x, y_vals, lw=2, color="tab:blue")
    ax.set_title(name)
    ax.axhline(0, color="grey", lw=0.8)
    ax.axvline(0, color="grey", lw=0.8)
    ax.grid(alpha=0.3)
    ax.set_xlabel("x")

plt.tight_layout()
plt.show()

### Exercise 2 — implement ReLU and sigmoid

Write both so they work on a whole NumPy array at once — no Python loops. This matters: the
loop version is correct but roughly a hundred times slower, and you will call these functions
millions of times.

* ReLU: `np.maximum(0, x)` compares element by element and keeps the larger.
* Sigmoid: `1 / (1 + np.exp(-x))`.


In [ ]:
# TODO: return max(0, x) for every element, with no Python loop
def relu(x):
    ...


# TODO: return 1 / (1 + e^(-x)) for every element
def sigmoid(x):
    ...


# --- self-check ---
test = np.array([-3.0, -0.5, 0.0, 0.5, 3.0])

assert np.allclose(relu(test), [0, 0, 0, 0.5, 3.0]), f"relu wrong: {relu(test)}"
assert np.allclose(sigmoid(0.0), 0.5), "sigmoid(0) must be exactly 0.5"
assert np.allclose(sigmoid(test),
                   [0.04742587, 0.37754067, 0.5, 0.62245933, 0.95257413]), \
    f"sigmoid wrong: {sigmoid(test)}"
assert relu(np.zeros((2, 3))).shape == (2, 3), "must work on 2-D arrays too"
print("Exercise 2 passed")

print("\nrelu   :", relu(test))
print("sigmoid:", sigmoid(test))

---
## Idea 3 — How a bend becomes a curve

ReLU looks far too simple to be powerful. One bend at zero — how does that let a network fit
anything complicated?

Here is the trick. Take **two** ReLU neurons. The first switches on at some point; the second
switches on later and subtracts. Between the two switch points, you get a ramp. After both are
on, they cancel out and you get a flat line.

Combine three shifted ReLUs and you get a clean **triangle**:

```
relu(x + 1)  -  2 * relu(x)  +  relu(x - 1)
```


In [ ]:
x = np.linspace(-3, 3, 400)

def relu_np(v):
    return np.maximum(0, v)

bump = relu_np(x + 1) - 2 * relu_np(x) + relu_np(x - 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].plot(x, relu_np(x + 1), "--", label="relu(x+1)")
axes[0].plot(x, -2 * relu_np(x), "--", label="-2*relu(x)")
axes[0].plot(x, relu_np(x - 1), "--", label="relu(x-1)")
axes[0].set_title("Three ReLUs...")
axes[0].legend(fontsize=8)

axes[1].plot(x, bump, lw=2.5, color="crimson")
axes[1].set_title("...added together make a bump")

for ax in axes:
    ax.axhline(0, color="grey", lw=0.8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Now the payoff. If you can build one bump, you can build many bumps, at any position, of any
height. And with enough small bumps you can trace **any** curve.

The cell below builds a sine wave out of nothing but ReLUs — no `np.sin` in the output at all,
just a pile of bends added together.


In [ ]:
x = np.linspace(-np.pi, np.pi, 500)

centers = np.linspace(-np.pi, np.pi, 21)
h = centers[1] - centers[0]

approx = np.zeros_like(x)
for c in centers:
    # one unit-height triangle centred at c
    triangle = (relu_np(x - (c - h)) - 2 * relu_np(x - c) + relu_np(x - (c + h))) / h
    approx += np.sin(c) * triangle          # scale it to the height we want there

plt.figure(figsize=(9, 4))
plt.plot(x, np.sin(x), lw=4, alpha=0.35, color="green", label="true sin(x)")
plt.plot(x, approx, lw=1.8, color="crimson", label="built from ReLUs only")
plt.axhline(0, color="grey", lw=0.8)
plt.title("A curve made entirely out of bends")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

That red line contains no sine function. It is 21 triangles, each made from three ReLUs,
added up.

Here we placed and scaled every bump by hand. Training is the process of finding those positions
and heights automatically.

---
## Idea 4 — A layer you can actually reuse

Writing `np.dot(X, W.T) + b` by hand every time gets old fast, and it does not scale to a
network with ten layers. Time to wrap it in a class.

### One change first: we are flipping how weights are stored

Read this carefully, because it will confuse you at least once otherwise.

| | Lab 1 | From now on |
|---|---|---|
| Weight shape | `(n_neurons, n_inputs)` | `(n_inputs, n_neurons)` |
| Forward pass | `np.dot(X, W.T) + b` | `np.dot(X, W) + b` |

Same maths, same results. We just store the matrix already transposed, so the `.T` disappears
from the forward pass. This is what the textbook does, and it matches what PyTorch and
TensorFlow do internally. Making the transpose go away is worth the one-time confusion.

### Initialization

```python
self.weights = 0.1 * np.random.randn(n_inputs, n_neurons)
self.biases  = np.zeros((1, n_neurons))
```

Two deliberate choices:

* **Weights start small and random.** Random, because if every weight were identical, every
  neuron in a layer would compute the same thing forever and the layer would be pointless.
  Small, because large values make the network's outputs explode as they pass through layers.
* **Biases start at zero.** There is no symmetry problem to break — the random weights already
  handle that.

Note the bias shape `(1, n_neurons)`. That leading 1 lets NumPy **broadcast** the bias across
every row of the batch automatically — neuron 1's bias gets added to every sample's neuron-1
output, exactly as in Lab 1.


### Exercise 3 — build `Layer_Dense`

Fill in `__init__` and `forward`. This class is used everywhere below and in the Assessment, so
get it right before moving on.


In [ ]:
class Layer_Dense:

    def __init__(self, n_inputs, n_neurons):
        # TODO: small random weights, shape (n_inputs, n_neurons)
        #       use 0.1 * np.random.randn(...)
        self.weights = ...

        # TODO: zero biases, shape (1, n_neurons)
        self.biases = ...

    def forward(self, inputs):
        # TODO: inputs @ weights + biases.  No .T needed now.
        #       Store it on self.output AND return it.
        self.output = ...
        return self.output


# --- self-check ---
np.random.seed(0)
test_layer = Layer_Dense(4, 3)

assert test_layer.weights.shape == (4, 3), \
    f"weights should be (n_inputs, n_neurons) = (4, 3), got {test_layer.weights.shape}"
assert test_layer.biases.shape == (1, 3), \
    f"biases should be (1, n_neurons) = (1, 3), got {test_layer.biases.shape}"
assert np.all(test_layer.biases == 0), "biases must start at zero"

out = test_layer.forward(np.ones((5, 4)))
assert out.shape == (5, 3), f"5 samples in, 5 rows out - got {out.shape}"
assert test_layer.output is not None, "remember to store the result on self.output"
print("Exercise 3 passed")
print("\noutput shape for a batch of 5:", out.shape)

---
## Idea 5 — Softmax: turning scores into probabilities

Your output layer produces raw numbers — the book calls them **logits**. Something like
`[12, 99, 318]`. Class 3 wins, but the numbers themselves say nothing useful. How confident is
the network? You cannot tell.

**Softmax** converts any list of scores into a proper probability distribution:

* every value is between 0 and 1,
* the values add up to exactly 1,
* the biggest score stays the biggest.

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Two steps: **exponentiate** everything (which makes it all positive and exaggerates
differences), then **divide by the total** (which makes it sum to 1).

Now `[0.45, 0.55]` tells you something `[12, 14]` never could: the network picked class 2, but
it is barely sure. That is information you can act on.

### One catch: overflow

`np.exp(1000)` is infinity, and `inf / inf` is `nan`. Your network is dead and the error message
will not tell you why.

The fix is a small piece of algebra. Subtracting a constant from every score before
exponentiating leaves the result **completely unchanged** — the constant cancels between the
numerator and the denominator. So we subtract the largest value in each row. The biggest input
to `exp` becomes 0, `exp(0) = 1`, and nothing can overflow.

Always do this. Every real implementation does.


### Exercise 4 — implement softmax

Work **row by row**, since each sample needs its own distribution. `keepdims=True` preserves
the 2-D shape so broadcasting lines up.

```python
np.max(x, axis=1, keepdims=True)   # largest value in each row
np.sum(x, axis=1, keepdims=True)   # row totals
```


In [ ]:
def softmax(x):
    # TODO: subtract the row maximum from every element (overflow protection)
    shifted = ...

    # TODO: exponentiate
    exp_values = ...

    # TODO: divide each row by that row's sum
    probabilities = ...

    return probabilities


# --- self-check ---
out = softmax(np.array([[1.0, 2.0, 3.0]]))
assert np.allclose(out, [[0.09003057, 0.24472847, 0.66524096]]), f"wrong values: {out}"
assert np.allclose(np.sum(out), 1.0), "each row must sum to 1"

batch = softmax(np.array([[1.0, 2.0, 3.0],
                          [5.0, 1.0, 1.0]]))
assert np.allclose(np.sum(batch, axis=1), [1.0, 1.0]), "every row must sum to 1"

# The overflow test - this is what the max-subtraction is for.
big = softmax(np.array([[1000.0, 1001.0, 1002.0]]))
assert not np.any(np.isnan(big)), "you got nan - did you subtract the row max?"
assert np.allclose(big, out), "shifting all scores by a constant must not change the result"
print("Exercise 4 passed")

print("\nscores [1, 2, 3]       ->", out[0])
print("scores [1000,1001,1002] ->", big[0], " (identical - only differences matter)")

---
## Idea 6 — Loss: putting a number on "wrong"

The network now outputs probabilities. To improve it, you need a single number saying how bad
those probabilities are. That number is the **loss**.

For classification we use **categorical cross-entropy**. Despite the name, when the true label
is a single class the formula reduces to something very simple:

$$\text{loss} = -\log(\text{probability assigned to the correct class})$$

That is the whole thing. Look only at the confidence given to the *right* answer, take the
negative log, done.

| Confidence in correct class | Loss | Reading |
|---|---|---|
| 1.00 | 0.00 | perfect |
| 0.90 | 0.11 | good |
| 0.50 | 0.69 | unsure |
| 0.10 | 2.30 | bad |
| 0.01 | 4.61 | confidently wrong — heavily punished |

The negative log does something valuable: it punishes **confident mistakes** far more than
uncertain ones. Being 1% sure of the right answer costs you forty times more than being 90%
sure. A model that is confidently wrong gets hurt, which is exactly the incentive you want.

### One catch: log(0) is infinity

If the network gives the correct class a probability of exactly 0, the loss is infinite and
training breaks permanently. The standard fix is to **clip** probabilities into
`[1e-7, 1 - 1e-7]` — close enough to 0 and 1 to be honest, far enough away to stay finite.

### Loss is not accuracy

Keep these separate in your head:

* **Accuracy** — the fraction of samples where the highest probability was on the right class.
  Easy to explain, but it moves in jumps and tells you nothing about near-misses.
* **Loss** — a smooth number that responds to every small improvement in confidence.

You *train* on loss because it is smooth. You *report* accuracy because people understand it.
Loss will often fall for a while before accuracy moves at all, and that is normal.


### Exercise 5 — implement categorical cross-entropy

`y_pred` has shape `(n_samples, n_classes)`; `y_true` is a flat array of correct class indices.

The key line picks out one probability per row:

```python
y_pred[range(len(y_pred)), y_true]
```

Row 0 takes column `y_true[0]`, row 1 takes column `y_true[1]`, and so on — one number per
sample, the confidence given to the correct class.


In [ ]:
def categorical_crossentropy(y_pred, y_true):
    # TODO: clip to avoid log(0). Use np.clip(y_pred, 1e-7, 1 - 1e-7)
    y_pred_clipped = ...

    # TODO: pick out the probability of the CORRECT class for each sample
    correct_confidences = ...

    # TODO: take -log of each, then return the MEAN over the batch
    return ...


# --- self-check ---
y_pred = np.array([[0.7,  0.1, 0.2 ],
                   [0.1,  0.5, 0.4 ],
                   [0.02, 0.9, 0.08]])
y_true = np.array([0, 1, 1])

loss = categorical_crossentropy(y_pred, y_true)
assert np.isscalar(loss) or loss.ndim == 0, "loss must be a single number, not an array"
assert abs(loss - 0.38506088) < 1e-6, f"expected 0.385061, got {loss}"

perfect = categorical_crossentropy(np.array([[1.0, 0.0, 0.0]]), np.array([0]))
assert perfect < 1e-5, "a perfect prediction should have almost zero loss"

confident_wrong = categorical_crossentropy(np.array([[0.0, 0.0, 1.0]]), np.array([0]))
assert confident_wrong > 10, "a confidently wrong prediction should be punished hard"
print("Exercise 5 passed")

print(f"\nloss             = {loss:.6f}")
print(f"perfect          = {perfect:.6f}")
print(f"confidently wrong= {confident_wrong:.6f}   <- and this is WITH clipping")

---
## Idea 7 — Which way is downhill?

You can now measure how wrong the network is. The remaining question is what to *do* about it.

You have one weight and a loss. Should you increase the weight or decrease it? What you need is
the **slope** of the loss as that weight changes:

* slope **positive** → increasing the weight increases the loss → go the other way
* slope **negative** → increasing the weight decreases the loss → keep going
* slope **zero** → you are at the bottom, or at least somewhere flat

The slope is the **derivative**. You do not need to have taken calculus to compute one — you can
measure it by nudging:

$$f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}$$

Move a tiny step right, a tiny step left, see how much the output changed, divide by the
distance travelled. With a small `h` this is very accurate. It is called the **numerical
derivative**, and it is the honest, brute-force way to get a slope.

When you have many parameters, the collection of all their individual slopes is called the
**gradient**. It is just a list of derivatives — one per parameter — and it points in the
direction that increases the loss fastest. So to *decrease* the loss you move the opposite way.

That single sentence is all of gradient descent.


### Exercise 6 — measure a slope

Implement the formula above. `f` is any function that takes one number and returns one number.


In [ ]:
def numerical_derivative(f, x, h=1e-5):
    """Approximate f'(x) by nudging x a tiny amount either side."""
    # TODO: (f(x+h) - f(x-h)) / (2*h)
    return ...


# --- self-check ---
# f(x) = x^2  ->  f'(x) = 2x  ->  f'(3) = 6
assert abs(numerical_derivative(lambda x: x**2, 3.0) - 6.0) < 1e-4

# f(x) = x^2 - 4x + 5  ->  f'(x) = 2x - 4  ->  f'(8) = 12
assert abs(numerical_derivative(lambda x: x**2 - 4*x + 5, 8.0) - 12.0) < 1e-4

# a flat line has zero slope everywhere
assert abs(numerical_derivative(lambda x: 7.0, 2.0)) < 1e-4
print("Exercise 6 passed")

# Where is the slope zero? That is the bottom of the curve.
f = lambda x: x**2 - 4*x + 5
for x_val in [0.0, 1.0, 2.0, 3.0, 4.0]:
    d = numerical_derivative(f, x_val)
    direction = "go right" if d < 0 else ("go left" if d > 0 else "at the bottom")
    print(f"x = {x_val}   f(x) = {f(x_val):5.2f}   slope = {d:7.3f}   -> {direction}")

---
## Idea 8 — Gradient descent

You have a slope. Now use it, repeatedly:

```
new_value = old_value  -  learning_rate * slope
```

The **minus** is the whole point. The slope points uphill, and you want to go downhill.

The **learning rate** controls step size, and it is the most important number you will ever
choose by hand:

* **too small** — you crawl, and training takes forever
* **too large** — you overshoot the bottom, bounce between the walls, and may diverge entirely
* **about right** — you converge quickly and settle

Watch it happen. We will minimise $f(x) = x^2 - 4x + 5$, whose minimum is at $x = 2$.


### Exercise 7 — descend the curve

Start at `x = 8.0` and take 50 steps.


In [ ]:
f = lambda x: x**2 - 4*x + 5

x = 8.0
learning_rate = 0.1
path = [x]

for step in range(50):
    # TODO: get the slope at the current x (use numerical_derivative)
    slope = ...

    # TODO: step DOWNHILL and store the new x in `x`
    x = ...

    path.append(x)

print(f"started at x = 8.0,  f(x) = {f(8.0):.4f}")
print(f"ended   at x = {x:.4f},  f(x) = {f(x):.4f}")

# --- self-check ---
assert abs(x - 2.0) < 0.01, \
    f"should converge to x = 2, got {x}. If it exploded, check the minus sign."
assert f(x) < f(8.0), "the function value must go DOWN"
print("Exercise 7 passed")

Now see the path it took, and what happens with a badly chosen learning rate.

In [ ]:
curve_x = np.linspace(-1, 9, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

for ax, lr in zip(axes, [0.01, 0.1, 0.95]):
    xi = 8.0
    pts = [xi]
    for _ in range(50):
        xi = xi - lr * numerical_derivative(f, xi)
        pts.append(xi)

    ax.plot(curve_x, f(curve_x), color="lightgrey", lw=2)
    ax.plot(pts, [f(p) for p in pts], "o-", ms=4, lw=1, color="crimson", alpha=0.8)
    ax.axvline(2, color="green", ls="--", lw=1, label="true minimum")
    ax.set_title(f"learning rate = {lr}\nended at x = {pts[-1]:.3f}")
    ax.set_xlabel("x")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel("f(x)")
plt.tight_layout()
plt.show()

Three learning rates, same starting point, same 50 steps:

* **0.01** — still nowhere near the bottom. Correct direction, far too slow.
* **0.1** — lands neatly at the minimum.
* **0.95** — overshoots past the bottom every single time and zig-zags across the valley.

Choosing this number is a real part of the job. Lab 5 covers optimisers that adapt it for you.


---
## Idea 9 — Backpropagation

Numerical derivatives work. So why not just use them for everything?

Count the cost. Each parameter needs two extra forward passes. A network with 100,000
parameters therefore needs **200,000 forward passes** to compute one gradient — for a single
update step. Training would take years.

**Backpropagation** computes every one of those derivatives *analytically*, in a single
backward sweep, for roughly the cost of one forward pass. It is the algorithm that makes deep
learning possible at all.

### The chain rule, in one line

A neuron is a chain of small operations:

```
multiply  ->  add up  ->  add bias  ->  ReLU
```

To find out how much a weight affects the final output, you multiply the local effects along
the chain backwards. That is the **chain rule**:

$$\frac{d}{dx}f(g(x)) = f'(g(x)) \cdot g'(x)$$

You need only three local derivatives, and every one is trivial:

| Operation | Local derivative | In words |
|---|---|---|
| `z = a * b` | w.r.t. `a` it is `b` | the other factor |
| `z = a + b` | it is `1` | addition passes the gradient straight through |
| `y = relu(z)` | `1` if `z > 0` else `0` | a gate: open or shut |

### Worked example

Take the neuron from the textbook:

```
x = [ 1.0, -2.0, 3.0]     inputs
w = [-3.0, -1.0, 2.0]     weights
b = 1.0                   bias
```

**Forward:** `z = (1)(-3) + (-2)(-1) + (3)(2) + 1 = 6.0`, then `y = relu(6.0) = 6.0`.

**Backward.** Start with a gradient of 1 at the output and walk back:

1. Through ReLU: `z = 6 > 0`, so the gate is **open** and the gradient passes through unchanged.
2. Through the sum: addition passes gradients straight through to all four terms.
3. Through each multiplication: the derivative with respect to `w[i]` is the *other factor*,
   which is `x[i]`.

So `dw = x = [1, -2, 3]`, `dx = w = [-3, -1, 2]`, and `db = 1`. No calculus beyond
"the derivative of a product is the other thing".


### Exercise 8 — backward pass through a neuron

Compute the gradients by hand, then check them against `numerical_derivative`. If your chain
rule is right, the two will agree to several decimal places.


In [ ]:
x = [ 1.0, -2.0, 3.0]
w = [-3.0, -1.0, 2.0]
b = 1.0

# ---------- forward pass ----------
z = x[0]*w[0] + x[1]*w[1] + x[2]*w[2] + b
y = max(z, 0)
print(f"forward:  z = {z},  y = relu(z) = {y}")

# ---------- backward pass ----------
dvalue = 1.0                      # gradient arriving from above

# TODO: through ReLU. Gate is open (pass dvalue) if z > 0, else closed (0).
drelu = ...

# TODO: gradient w.r.t. each weight. For w[i] the local derivative is x[i],
#       multiplied by drelu from the chain rule.
dw = ...

# TODO: gradient w.r.t. each input. For x[i] the local derivative is w[i].
dx = ...

# TODO: gradient w.r.t. the bias. Addition passes the gradient straight through.
db = ...

print(f"\nbackward: dw = {dw}")
print(f"          dx = {dx}")
print(f"          db = {db}")

# --- self-check against numerical derivatives ---
def neuron_output(w0, w1, w2, bias):
    zz = x[0]*w0 + x[1]*w1 + x[2]*w2 + bias
    return max(zz, 0)

num_dw0 = numerical_derivative(lambda v: neuron_output(v, w[1], w[2], b), w[0])
num_db  = numerical_derivative(lambda v: neuron_output(w[0], w[1], w[2], v), b)

print(f"\nchain rule dw[0] = {dw[0]:.5f}   numerical = {num_dw0:.5f}")
print(f"chain rule db    = {db:.5f}   numerical = {num_db:.5f}")

assert np.allclose(dw, [1.0, -2.0, 3.0]), f"dw wrong: {dw}"
assert np.allclose(dx, [-3.0, -1.0, 2.0]), f"dx wrong: {dx}"
assert abs(db - 1.0) < 1e-9, f"db wrong: {db}"
assert abs(dw[0] - num_dw0) < 1e-4, "chain rule disagrees with the numerical check"
print("\nExercise 8 passed - hand-derived gradients match the numerical ones.")

Now use those gradients. Take one small step downhill and check the output really did drop.


In [ ]:
print(f"output before: {y}")

learning_rate = 0.001
w = [w[i] - learning_rate * dw[i] for i in range(3)]
b = b - learning_rate * db

z_new = x[0]*w[0] + x[1]*w[1] + x[2]*w[2] + b
y_new = max(z_new, 0)

print(f"output after : {y_new}")
print(f"change       : {y_new - y:.4f}   <- it went down, exactly as intended")

That is a complete learning step: forward, measure, backward, update.

Everything from here is scale. Instead of one neuron you have millions, instead of minimising
a neuron's output you minimise the loss, and instead of writing the chain rule by hand a
framework does it for you. The mechanism does not change.


---

# Assessment

**Marks: 1**

Everything you built today, assembled into a network that trains itself.

```
X (300, 2)  ->  Dense(2, 8)  ->  ReLU  ->  Dense(8, 3)  ->  Softmax  ->  loss
```

Run the setup cell first. It creates the dataset and gives you a `numerical_gradient` helper —
that part is mechanical plumbing based on Exercise 6, so it is provided.


In [ ]:
# --- Assessment setup: DO NOT MODIFY ---

np.random.seed(0)
X_train, y_train = vertical_data(samples=100, classes=3)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape, " classes:", np.unique(y_train))

plt.figure(figsize=(5, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=18, cmap="brg")
plt.title("The dataset: 3 classes, 2 features")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.show()


def numerical_gradient(param, loss_fn, h=1e-4):
    """Gradient of loss_fn() with respect to every element of `param`.

    Nudges each element in turn, exactly like Exercise 6, and puts the
    parameter back afterwards.
    """
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        original = param[idx]

        param[idx] = original + h
        loss_plus = loss_fn()

        param[idx] = original - h
        loss_minus = loss_fn()

        param[idx] = original                       # restore
        grad[idx] = (loss_plus - loss_minus) / (2 * h)
        it.iternext()
    return grad


print("\nSetup done.")

### Q1 — Build the network

Create the two layers, then write `forward(inputs)` that runs the full chain and returns the
**probabilities**.

Use your own `Layer_Dense`, `relu` and `softmax` from earlier in this lab.


In [ ]:
np.random.seed(42)

# TODO: create the two dense layers.
#       Layer 1: 2 features in, 8 neurons out
#       Layer 2: 8 inputs in,   3 classes out
dense1 = ...
dense2 = ...


def forward(inputs):
    """Dense -> ReLU -> Dense -> Softmax. Returns probabilities."""
    # TODO: run the four steps and return the final probabilities
    ...


# --- self-check ---
probs = forward(X_train)
assert probs.shape == (300, 3), f"expected (300, 3), got {probs.shape}"
assert np.allclose(np.sum(probs, axis=1), 1.0), "every row must sum to 1 (softmax)"
assert dense1.weights.shape == (2, 8), f"dense1 weights should be (2, 8)"
assert dense2.weights.shape == (8, 3), f"dense2 weights should be (8, 3)"
print("Q1 passed")
print("\nfirst 3 samples' probabilities:")
print(probs[:3])

### Q2 — Loss and accuracy before training

Compute both for the untrained network.

Accuracy is the fraction of samples whose highest probability sits on the correct class:
`np.mean(np.argmax(probs, axis=1) == y_train)`.


In [ ]:
# TODO: loss on the untrained network (use your categorical_crossentropy)
start_loss = ...

# TODO: accuracy on the untrained network
start_acc = ...

print(f"starting loss     : {start_loss:.4f}")
print(f"starting accuracy : {start_acc:.4f}")

# --- self-check ---
assert 1.0 < start_loss < 1.3, \
    f"an untrained 3-class model should start near ln(3) = 1.0986, got {start_loss}"
assert 0.0 <= start_acc <= 1.0, "accuracy must be a fraction between 0 and 1"
print("\nQ2 passed")

The starting loss should be very close to **1.0986**. That is not a coincidence — it is
`ln(3)`. An untrained network spreads its probability evenly across the 3 classes, giving each
one about 0.333, and `-log(1/3) = 1.0986`. If your starting loss is far from this, something in
the forward pass is wrong.

### Q3 — Train it

Write the training loop. Each epoch:

1. record the current loss and accuracy,
2. compute the gradient for every parameter with `numerical_gradient`,
3. update each parameter: `param -= learning_rate * grad`.

Update the arrays **in place** with `-=`. Writing `param = param - ...` creates a new array and
leaves the layer's actual weights untouched, so nothing will learn.

This takes a few seconds to run.


In [ ]:
def loss_fn():
    """Current loss. numerical_gradient calls this repeatedly."""
    return categorical_crossentropy(forward(X_train), y_train)


params = [dense1.weights, dense1.biases, dense2.weights, dense2.biases]
learning_rate = 1.0
epochs = 150

loss_history = []
acc_history = []

for epoch in range(epochs):
    # TODO: record the current loss and accuracy
    loss_history.append(...)
    acc_history.append(...)

    # TODO: gradient for every parameter array
    grads = [... for p in params]

    # TODO: update each parameter IN PLACE, downhill
    for p, g in zip(params, grads):
        ...

    if epoch % 30 == 0:
        print(f"epoch {epoch:3d}   loss {loss_history[-1]:.4f}   acc {acc_history[-1]:.4f}")

final_loss = loss_fn()
final_acc = np.mean(np.argmax(forward(X_train), axis=1) == y_train)
print(f"\nfinal        loss {final_loss:.4f}   acc {final_acc:.4f}")

# --- self-check ---
assert len(loss_history) == epochs, f"expected {epochs} recorded losses"
assert final_loss < 0.5, \
    f"loss should fall well below 0.5, got {final_loss}. Check the minus sign and use -=."
assert final_acc > 0.80, f"accuracy should exceed 0.80, got {final_acc}"
print("Q3 passed")

Plot the two curves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].plot(loss_history, color="crimson", lw=2)
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].axhline(np.log(3), color="grey", ls="--", lw=1, label="ln(3) = random guessing")
axes[0].legend(fontsize=8)

axes[1].plot(acc_history, color="tab:blue", lw=2)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylim(0, 1)
axes[1].axhline(1/3, color="grey", ls="--", lw=1, label="1/3 = random guessing")
axes[1].legend(fontsize=8)

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## What you built today

A network that **learns**. It started at random-guessing accuracy and taught itself to
classify, using nothing but NumPy and ideas you can hold in your head:

* activations, because otherwise depth is decoration,
* loss, because you cannot improve what you cannot measure,
* gradients, because they tell you which way is downhill,
* the chain rule, because it makes gradients affordable.


### Before next week

* Restart the kernel and run everything top to bottom. Every assertion should pass.
* Keep this notebook. `Layer_Dense`, `relu`, `softmax` and `categorical_crossentropy` all come
  back, and you will not want to rewrite them.
* If the chain rule in Idea 9 felt shaky, redo Exercise 8 on paper with different numbers and
  check yourself with `numerical_derivative`.
* Book reference: Chapters 4 to 9.

---

### Submission
   
1. Create a new folder in your course repository on GitHub for this lab.
2. Rename the notebook to: `arti402_Lab2_<YourID>.ipynb`
3. Upload the completed notebook to the folder you created. Make sure that all outputs are visible before uploading.

**End of notebook.**